# Cognitive Load Analysis from Keystroke Dynamics

This notebook analyzes the collected keystroke data to determine cognitive load levels across different tasks.

**Dataset**: Keystroke features with three cognitive load levels (Low, Medium, High)

**Analysis Goals**:
1. Explore keystroke patterns by cognitive load level
2. Statistical analysis of features
3. Machine learning classification models
4. Feature importance analysis
5. NASA-TLX correlation

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All libraries imported successfully")

## 2. Load and Explore the Dataset

In [ ]:
# Load the keystroke features dataset
data_path = 'keystroke-data-collector/exports/Keystroke_Features_20251211_183126.csv'
df = pd.read_csv(data_path, sep=';')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Basic dataset information
print("Dataset Info:")
print(df.info())
print("\n" + "="*50)
print("\nCognitive Load Distribution:")
print(df['cognitiveLoadLevel'].value_counts())
print("\n" + "="*50)
print("\nDescriptive Statistics:")
df.describe()

## 3. Data Preprocessing and Cleaning

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
print("\n" + "="*50)

# Handle missing values - fill with 0 for timing features (indicates no data/event)
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(0)

# Verify no missing values remain
print(f"\nTotal missing values after cleaning: {df.isnull().sum().sum()}")

# Check data types
print("\n" + "="*50)
print("\nData types:")
print(df.dtypes)

## 4. Feature Engineering for Cognitive Load Indicators

In [ ]:
# Select keystroke timing features for analysis
timing_features = [
    'holdTime_mean', 'holdTime_std', 'holdTime_min', 'holdTime_max',
    'flightTime_D1U2_mean', 'flightTime_D1U2_std',
    'flightTime_D1D2_mean', 'flightTime_D1D2_std',
    'flightTime_U1D2_mean', 'flightTime_U1D2_std',
    'flightTime_U1U2_mean', 'flightTime_U1U2_std',
    'pause_count', 'pause_mean', 'pause_std', 'pause_max',
    'backspaceCount', 'errorCorrections', 'correctionRate',
    'totalKeystrokes', 'duration'
]

# Create additional derived features
df['typing_variability'] = df['holdTime_std'] / (df['holdTime_mean'] + 1)  # Avoid division by zero
df['pause_rate'] = df['pause_count'] / (df['duration'] + 1)
df['error_rate'] = df['errorCorrections'] / (df['totalKeystrokes'] + 1)
df['typing_speed'] = df['totalKeystrokes'] / (df['duration'] + 1) * 1000  # keystrokes per second

# Add NASA-TLX average
nasa_tlx_cols = ['mentalDemand', 'physicalDemand', 'temporalDemand', 'performance', 'effort', 'frustration']
df['nasa_tlx_avg'] = df[nasa_tlx_cols].mean(axis=1)

print("✓ Feature engineering complete")
print(f"\nNew features created: typing_variability, pause_rate, error_rate, typing_speed, nasa_tlx_avg")
print(f"\nTotal features available: {len(timing_features) + 5}")

## 5. Statistical Analysis of Cognitive Load Metrics

In [ ]:
# Group by cognitive load level and compute statistics
cognitive_groups = df.groupby('cognitiveLoadLevel')

# Key metrics to analyze
key_metrics = ['holdTime_mean', 'flightTime_D1U2_mean', 'pause_count', 'pause_mean', 
               'errorCorrections', 'typing_speed', 'typing_variability', 'nasa_tlx_avg']

print("Descriptive Statistics by Cognitive Load Level:")
print("="*70)
for metric in key_metrics:
    print(f"\n{metric}:")
    print(cognitive_groups[metric].agg(['mean', 'std', 'min', 'max']))
    
    # Perform ANOVA test
    groups = [group[metric].values for name, group in cognitive_groups]
    f_stat, p_value = stats.f_oneway(*groups)
    print(f"ANOVA: F-statistic={f_stat:.4f}, p-value={p_value:.4f}")
    if p_value < 0.05:
        print("✓ Significant difference between cognitive load levels")
    else:
        print("✗ No significant difference")

## 6. Visualize Cognitive Load Patterns

In [ ]:
# Cognitive load distribution
plt.figure(figsize=(10, 5))
df['cognitiveLoadLevel'].value_counts().sort_index().plot(kind='bar', color=['green', 'orange', 'red'])
plt.title('Cognitive Load Level Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Cognitive Load Level')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots for key features by cognitive load
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

features_to_plot = ['holdTime_mean', 'pause_count', 'errorCorrections', 
                    'typing_speed', 'typing_variability', 'nasa_tlx_avg']

for idx, feature in enumerate(features_to_plot):
    sns.boxplot(data=df, x='cognitiveLoadLevel', y=feature, 
                palette=['green', 'orange', 'red'], ax=axes[idx])
    axes[idx].set_title(f'{feature} by Cognitive Load', fontweight='bold')
    axes[idx].set_xlabel('Cognitive Load Level')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for all timing features
plt.figure(figsize=(14, 10))
feature_cols = timing_features + ['typing_variability', 'pause_rate', 'error_rate', 'typing_speed', 'nasa_tlx_avg']
correlation_matrix = df[feature_cols].corr()
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Machine Learning Classification Models

In [ ]:
# Prepare data for machine learning
X = df[timing_features + ['typing_variability', 'pause_rate', 'error_rate', 'typing_speed']]
y = df['cognitiveLoadLevel']

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"\nClass distribution in training set:")
print(pd.Series(y_train).value_counts().sort_index())
print(f"\nClass labels: {le.classes_}")

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42),
    'XGBoost': xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss')
}

# Train and evaluate models
results = {}
for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"Training {name}...")
    
    # Train model
    model.fit(X_train_scaled, y_train)
    
    # Predictions
    y_pred = model.predict(X_test_scaled)
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = {
        'accuracy': accuracy,
        'predictions': y_pred,
        'model': model
    }
    
    print(f"\nAccuracy: {accuracy:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=le.classes_))
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title(f'{name} - Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()

print(f"\n{'='*60}")
print("\nModel Comparison:")
for name, result in results.items():
    print(f"{name}: {result['accuracy']:.4f}")

## 8. Feature Importance Analysis

In [ ]:
# Feature importance from Random Forest
rf_model = results['Random Forest']['model']
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 Most Important Features (Random Forest):")
print(feature_importance.head(10))

# Visualize feature importance
plt.figure(figsize=(12, 8))
top_features = feature_importance.head(15)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance')
plt.title('Top 15 Feature Importance for Cognitive Load Classification', fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 9. NASA-TLX Correlation Analysis

In [ ]:
# NASA-TLX dimensions by cognitive load
plt.figure(figsize=(14, 6))
nasa_tlx_by_load = df.groupby('cognitiveLoadLevel')[nasa_tlx_cols].mean()
nasa_tlx_by_load.T.plot(kind='bar', color=['green', 'orange', 'red'])
plt.title('NASA-TLX Dimensions by Cognitive Load Level', fontsize=14, fontweight='bold')
plt.xlabel('NASA-TLX Dimension')
plt.ylabel('Average Score (0-100)')
plt.legend(title='Cognitive Load')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nNASA-TLX Statistics by Cognitive Load:")
print(nasa_tlx_by_load)

In [ ]:
# Correlation between keystroke features and NASA-TLX
features_for_correlation = ['holdTime_mean', 'pause_count', 'errorCorrections', 
                            'typing_speed', 'typing_variability']
nasa_features = nasa_tlx_cols + ['nasa_tlx_avg']

correlation_data = df[features_for_correlation + nasa_features].corr()
keystroke_nasa_corr = correlation_data.loc[features_for_correlation, nasa_features]

plt.figure(figsize=(10, 6))
sns.heatmap(keystroke_nasa_corr, annot=True, cmap='RdYlGn_r', center=0, 
            fmt='.2f', square=False, cbar_kws={"shrink": 0.8})
plt.title('Correlation: Keystroke Features vs NASA-TLX Dimensions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey Correlations with NASA-TLX Average:")
nasa_corr = keystroke_nasa_corr['nasa_tlx_avg'].sort_values(ascending=False)
print(nasa_corr)

## 10. Summary and Conclusions

In [ ]:
print("="*70)
print("COGNITIVE LOAD ANALYSIS SUMMARY")
print("="*70)

print(f"\n📊 Dataset Overview:")
print(f"   • Total Sessions: {len(df)}")
print(f"   • Participants: {df['userId'].nunique()}")
print(f"   • Cognitive Load Distribution:")
for level in ['Low', 'Medium', 'High']:
    count = (df['cognitiveLoadLevel'] == level).sum()
    print(f"     - {level}: {count} sessions")

print(f"\n🤖 Model Performance:")
best_model = max(results.items(), key=lambda x: x[1]['accuracy'])
print(f"   • Best Model: {best_model[0]}")
print(f"   • Accuracy: {best_model[1]['accuracy']:.4f}")

print(f"\n🔑 Top 5 Most Important Features:")
for idx, row in feature_importance.head(5).iterrows():
    print(f"   {idx+1}. {row['feature']}: {row['importance']:.4f}")

print(f"\n📈 Key Findings:")
print(f"   • Keystroke dynamics show measurable differences across cognitive load levels")
print(f"   • Machine learning models can classify cognitive load with reasonable accuracy")
print(f"   • Feature importance reveals which keystroke patterns are most indicative")
print(f"   • NASA-TLX scores correlate with objective keystroke metrics")

print("\n" + "="*70)